# B02 辅修：补全一次更新并比较学习率

学生练习稿 · 2 学时。数据为合成，不作真实业务或因果推断。

只使用训练与验证，无最终测试结果。J=mean(e²)/2，RMSE=sqrt(2J)。

请先写姓名与日期，并在给定时间内完成核心任务。学生稿含 TODO，填完后再从头运行；停止提示不是环境故障。

## 1. 读取固定数据
环境已由教师准备；在非保护盘个人目录保存整个单元副本。production 指开工前已确定的计划产量，energy 为完工后能耗。

本单元手写一元公式使用一维 x；B01 的库输入 X 保留二维特征轴。需要调用库时用 x[:, None] 转为 (n,1)，两种约定不要混算。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "data/regression.csv").is_file()), None)
if ROOT is None:
    ROOT = next(
        (
            p / "units/linear-regression"
            for p in candidates
            if (p / "units/linear-regression/data/regression.csv").is_file()
        ),
        None,
    )
if ROOT is None:
    raise FileNotFoundError(
        "请从课程根目录、单元目录或 labs/teacher 目录启动，保留完整数据目录。"
    )
df = pd.read_csv(ROOT / "data/regression.csv")
tr, va = df[df["split"] == "train"], df[df["split"] == "validation"]
xt, yt = tr["production"].to_numpy(), tr["energy"].to_numpy()
xv, yv = va["production"].to_numpy(), va["energy"].to_numpy()
mu, sigma = xt.mean(), xt.std(ddof=0)
zt, zv = (xt - mu) / sigma, (xv - mu) / sigma
assert len(tr) == 80 and len(va) == 40 and sigma > 0
print("训练/验证:", len(tr), len(va), "标准化参数仅来自训练部分")
display(df.head())

## 2. 读懂已给函数
指出预测、残差平方和平均分别对应哪个表达式。

In [ ]:
def predict(x, w, b):
    return w * x + b


def half_mse(x, y, w, b):
    assert x.ndim == y.ndim == 1 and x.shape == y.shape
    return float(np.mean((predict(x, w, b) - y) ** 2) / 2)

In [ ]:
paper_x = np.array([0.0, 1.0, 2.0])
paper_y = np.array([1.0, 3.0, 5.0])
np.testing.assert_allclose(predict(paper_x, 1.0, 0.0), [0.0, 1.0, 2.0])
np.testing.assert_allclose(half_mse(paper_x, paper_y, 1.0, 0.0), 7 / 3)
np.testing.assert_allclose(half_mse(paper_x, paper_y, 2.0, 1.0), 0.0)
print("与 T03 手算一致")

## 3. 梯度和同步更新
先回忆 B1/B2：在旧参数上求两个梯度，再同时更新。

In [ ]:
def gradient(x, y, w, b):
    error = predict(x, w, b) - y
    # TODO: 根据 dw=mean(e*x)、db=mean(e) 返回两个标量
    raise NotImplementedError("请补全两个梯度")


def step(w, b, dw, db, learning_rate):
    # TODO: 用减号更新两个参数，返回新 w 和 b
    raise NotImplementedError("请补全更新")

In [ ]:
dw, db = gradient(paper_x, paper_y, 1.0, 0.0)
np.testing.assert_allclose([dw, db], [-8 / 3, -2.0])
w1, b1 = step(1.0, 0.0, dw, db, 0.1)
np.testing.assert_allclose([w1, b1], [19 / 15, 1 / 5])
np.testing.assert_allclose(half_mse(paper_x, paper_y, w1, b1), 1829 / 1350)
print("更新前 J=", 7 / 3, "更新后 J=", half_mse(paper_x, paper_y, w1, b1))

## 4. 只改变学习率
运行前写预测。两次均从零初始化、使用相同标准化训练数据和 200 次更新预算。验证数据不参与梯度。完整循环由教师提供。

In [ ]:
# 教师提供的训练组织代码。两个候选均从 (0,0) 开始；验证结果只记录，不参与梯度。
def fit_gd(x, y, x_val, y_val, learning_rate, steps=200):
    w, b = 0.0, 0.0
    rows = []
    status = "budget_completed"
    for iteration in range(steps + 1):
        j = half_mse(x, y, w, b)
        j_val = half_mse(x_val, y_val, w, b)
        rows.append([iteration, j, j_val, w, b])
        if not np.isfinite(j) or j > 1e12:
            status = "stopped_large_loss"
            break
        if iteration < steps:
            dw, db = gradient(x, y, w, b)
            w, b = step(w, b, dw, db, learning_rate)
    history = pd.DataFrame(rows, columns=["step", "train_J", "validation_J", "w", "b"])
    return history, status

In [ ]:
histories, rows = {}, []
for lr in [0.001, 0.1]:
    h, status = fit_gd(zt, yt, zv, yv, learning_rate=lr, steps=200)
    histories[lr] = h
    last = h.iloc[-1]
    rows.append(
        {
            "method": f"GD eta={lr}",
            "updates": int(last["step"]),
            "status": status,
            "train_RMSE": np.sqrt(2 * last["train_J"]),
            "validation_RMSE": np.sqrt(2 * last["validation_J"]),
        }
    )
reference = LinearRegression().fit(zt[:, None], yt)
rows.append(
    {
        "method": "least-squares reference",
        "updates": None,
        "status": "reference",
        "train_RMSE": np.sqrt(np.mean((reference.predict(zt[:, None]) - yt) ** 2)),
        "validation_RMSE": np.sqrt(np.mean((reference.predict(zv[:, None]) - yv) ** 2)),
    }
)
rows.append(
    {
        "method": "train-mean baseline",
        "updates": None,
        "status": "reference",
        "train_RMSE": np.sqrt(np.mean((yt.mean() - yt) ** 2)),
        "validation_RMSE": np.sqrt(np.mean((yt.mean() - yv) ** 2)),
    }
)
results = pd.DataFrame(rows)
display(results)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for lr, h in histories.items():
    for ax, column in zip(axes, ["train_J", "validation_J"]):
        ax.semilogy(h["step"], h[column], label=f"eta={lr}")
        ax.set(xlabel="Completed updates", ylabel="Half MSE J", title=column)
        ax.legend()
fig.tight_layout()
plt.show()

## 结果解释与提交
1. 当前数据和预算下哪个候选下降更快？指出图表依据。
2. 列出比较时保持不变的条件；解释为何每次要重置参数。
3. 训练与验证 RMSE 相比基线怎样？为何不能把这称为最终测试性能？

在此写本人回答。保存这一个 Notebook，无需另交长报告；注明 AI 等辅助工具用途。